# Day 13 — Bounding Box Foundations

**Agri-AI Bootcamp** · Phase 4 of 5: *Object Detection & Yield* · Days 13–16

> Write a PyTorch IoU function and test it on apple annotations vs simulated predictions.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

Detection predicts where an object is as well as what it is. Boxes are stored as corners (x1, y1, x2, y2) or centre, width, height. IoU divides overlap area by union area: 1 is perfect, 0 is no overlap, and 0.5 is a common cutoff for a correct detection.

### 🌾 Agriculture example

A labelled apple box vs a predicted box shifted slightly: the overlap is large, so IoU is high and the detection counts as a hit.

> **Watch out:** Clamp intersection width and height at 0 for non-overlapping boxes, or IoU can go negative.

## 2. Setup

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
torch.set_printoptions(precision=3)

## 3. Data

In [ ]:
# ---- Hand-written annotations: 5 apples on one tree photo (pixels) ----------
# format: [x1, y1, x2, y2] = top-left corner, bottom-right corner
GT = torch.tensor([[ 50,  60, 110, 120],
                   [140,  80, 196, 140],
                   [230, 150, 290, 215],
                   [ 90, 200, 150, 262],
                   [300,  40, 352,  96]], dtype=torch.float32)
# a detector's output: 4 hits of varying quality + 1 false positive
PRED = torch.tensor([[ 55,  65, 112, 124],      # very close to apple 0
                     [150,  90, 205, 150],      # shifted from apple 1
                     [232, 152, 288, 212],      # tight on apple 2
                     [120, 230, 180, 290],      # only partly over apple 3
                     [ 20, 300,  70, 350]],     # nothing there
                    dtype=torch.float32)
SCORES = torch.tensor([0.95, 0.88, 0.91, 0.52, 0.44])
IMG_W, IMG_H = 400, 380
print("ground truth boxes:", GT.shape, "| predictions:", PRED.shape)

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
def iou(a,b):   # boxes as tensors [x1,y1,x2,y2]
    x1=torch.max(a[0],b[0]); y1=torch.max(a[1],b[1])
    x2=torch.min(a[2],b[2]); y2=torch.min(a[3],b[3])
    inter=(x2-x1).clamp(0)*(y2-y1).clamp(0)
    union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union
```

## 5. Worked solution

In [ ]:
# ---- Box format conversions --------------------------------------------------
def xyxy_to_cxcywh(b):
    x1, y1, x2, y2 = b.unbind(-1)
    return torch.stack([(x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1], -1)
def cxcywh_to_xyxy(b):
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx-w/2, cy-h/2, cx+w/2, cy+h/2], -1)

print("corners      :", GT[0].tolist())
print("centre/w/h   :", xyxy_to_cxcywh(GT[0]).tolist())
print("round trip ok:", torch.allclose(cxcywh_to_xyxy(xyxy_to_cxcywh(GT)), GT))

# ---- IoU from scratch --------------------------------------------------------
def iou(a, b):
    """a, b: tensors [x1,y1,x2,y2]. Returns intersection over union."""
    x1 = torch.max(a[0], b[0]); y1 = torch.max(a[1], b[1])
    x2 = torch.min(a[2], b[2]); y2 = torch.min(a[3], b[3])
    inter = (x2-x1).clamp(min=0) * (y2-y1).clamp(min=0)     # clamp: no negative overlap
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else torch.tensor(0.)

# sanity tests -- write these BEFORE you trust the function
box = torch.tensor([0., 0., 10., 10.])
assert abs(iou(box, box).item() - 1.0) < 1e-6,                     "identical boxes -> 1"
assert iou(box, torch.tensor([20., 20., 30., 30.])).item() == 0.0,  "disjoint -> 0 (not negative)"
assert abs(iou(box, torch.tensor([0., 0., 10., 5.])).item() - 0.5) < 1e-6, "half overlap -> 0.5"
print("\nIoU unit tests passed")

def iou_matrix(A, B):
    return torch.tensor([[iou(a, b) for b in B] for a in A])

M = iou_matrix(PRED, GT)
print("\nIoU matrix (rows = predictions, cols = ground truth):\n", M)

# ---- Match predictions to ground truth at a 0.5 threshold -------------------
THR = 0.5
print(f"\nthreshold {THR}:")
used = set()
tp = fp = 0
for i in SCORES.argsort(descending=True):          # highest confidence first
    best = int(M[i].argmax()); best_iou = M[i, best].item()
    if best_iou >= THR and best not in used:
        used.add(best); tp += 1
        print(f"  pred {i.item()} (score {SCORES[i]:.2f}) -> HIT  apple {best}, IoU {best_iou:.3f}")
    else:
        fp += 1
        print(f"  pred {i.item()} (score {SCORES[i]:.2f}) -> MISS best IoU {best_iou:.3f}")
fn = len(GT) - tp
print(f"\nTP {tp}  FP {fp}  FN {fn} | precision {tp/(tp+fp):.2f} recall {tp/(tp+fn):.2f}")

# ---- Draw it -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5.7))
ax.set_xlim(0, IMG_W); ax.set_ylim(IMG_H, 0); ax.set_facecolor("#eaf3e6")
for b in GT:
    ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                 fill=False, ec="green", lw=2))
for b, s in zip(PRED, SCORES):
    ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                 fill=False, ec="crimson", lw=1.6, ls="--"))
    ax.text(b[0], b[1]-4, f"{s:.2f}", color="crimson", fontsize=8)
ax.set_title("green = labelled apples, red dashed = predictions"); plt.show()

## 6. Your turn

1. Re-run the matching at IoU 0.75. How many hits survive a stricter standard?
2. Vectorise `iou_matrix` with broadcasting (no Python loops) and check it matches.
3. Compare your function against `torchvision.ops.box_iou`.

In [ ]:
# TODO 1: stricter threshold
# change THR to 0.75 and re-run the matching block

# TODO 2: vectorised IoU
def box_iou_vec(A, B):
    tl = torch.max(A[:, None, :2], B[None, :, :2])
    br = torch.min(A[:, None, 2:], B[None, :, 2:])
    inter = (br - tl).clamp(min=0).prod(-1)
    area = lambda b: (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])
    return inter / (area(A)[:, None] + area(B)[None, :] - inter)
print("matches loop version:", torch.allclose(box_iou_vec(PRED, GT), M, atol=1e-6))

# TODO 3: torchvision reference
from torchvision.ops import box_iou
print("matches torchvision:", torch.allclose(box_iou(PRED, GT), M, atol=1e-6))

## 7. Dataset and references

**Dataset:** [Hand-write 4 to 6 box coordinates for apples or weeds](https://colab.research.google.com/#create=true)

**References**

- [PyImageSearch](https://pyimagesearch.com)

**Course material:** PyImageSearch: bounding boxes explained

---
**Next:** Day 14 — One-Stage Object Detectors